In [2]:
# A100 setup
import subprocess, sys, zipfile
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0', 'scipy==1.15.3'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'bitsandbytes'])
with zipfile.ZipFile('/content/controlled_editing_code_a100.zip') as z:
    z.extractall('/content')
import torch
print('GPU:', torch.cuda.get_device_name(0))
assert 'A100' in torch.cuda.get_device_name(0)
print('A100 SOURCE READY')


GPU: NVIDIA A100-SXM4-40GB
A100 SOURCE READY


In [3]:
# A100 recovery: train all seeds, then download immediately.
import subprocess, sys, shutil
from pathlib import Path
from google.colab import files
root = Path('/content/cvpr2027-a100'); root.mkdir(exist_ok=True)
for seed in (17, 29, 41):
    run = root / f'seed-{seed}'
    with (root/f'seed-{seed}.log').open('a') as log:
        process = subprocess.Popen([sys.executable, '-u', '/content/train_controlled_editing.py', '--output', str(run), '--epochs', '3', '--seed', str(seed), '--resume'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    assert code == 0, f'Seed {seed} failed: {code}'
    print('COMPLETED SEED', seed, flush=True)
shutil.make_archive('/content/cvpr2027-a100', 'zip', root)
print('ALL THREE SEEDS COMPLETE; downloading')
files.download('/content/cvpr2027-a100.zip')


2026-09-13 11:20:25.755575: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-13 11:20:25.817518: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# Verify that all three saved adapters can be loaded for inference.
import json, gc, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from controlled_editing_data import SYSTEM
from train_controlled_editing import load_rows, score_action
from google.colab import files
results = []
for seed in (17, 29, 41):
    run = Path(f'/content/cvpr2027-a100/seed-{seed}')
    manifest = json.loads((run/'run_manifest.json').read_text())
    assert manifest['status'] == 'completed'
    tokenizer = AutoTokenizer.from_pretrained(run/'adapter')
    base = AutoModelForCausalLM.from_pretrained(manifest['model'], revision=manifest['model_revision'], device_map={'':0}, torch_dtype=torch.float16)
    model = PeftModel.from_pretrained(base, run/'adapter').eval()
    selected = {}
    for row in load_rows(run/'data'/'test.jsonl'):
        target = row['target']
        selected.setdefault((target['action'], target.get('attribute'), target.get('reason')), row)
    assert len(selected) == 6
    for row in selected.values():
        prompt = tokenizer.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':row['input']}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(generated[0,inputs.input_ids.shape[1]:], skip_special_tokens=True)
        score = score_action(row,text)
        results.append({'seed':seed,'id':row['id'],'prediction':text,'target':row['target'],'metrics':score})
        print(seed,row['id'],score['exact_action'],flush=True)
    del model, base, inputs, generated
    gc.collect(); torch.cuda.empty_cache()
report = {'scope':'saved-adapter reload smoke check; six actions per seed','n':len(results),'exact':sum(r['metrics']['exact_action'] for r in results),'results':results}
Path('/content/adapter-reload-check.json').write_text(json.dumps(report,indent=2))
assert report['exact'] == report['n'] == 18
print('ADAPTER RELOAD VERIFIED')
files.download('/content/adapter-reload-check.json')


FileNotFoundError: Cannot find file: /content/cvpr2027-a100.zip